# Exercice 01 – Comparer deux modèles sur 50 questions

## Objectif

Envoyer les mêmes 50 questions (fichier `01_questions.md`) à **deux modèles différents** servis par `llama-server`, enregistrer les réponses et les métriques, puis comparer :

- la **justesse** sur les questions fermées (vérification automatique),
- la **qualité** sur les questions ouvertes (lecture manuelle côte à côte),
- la **vitesse** (`tokens/s`), la **longueur** des réponses et le **temps total**.

## Ce que vous allez pratiquer

- Une boucle d'appels à l'API avec le SDK OpenAI
- La lecture des champs `usage` et `timings` de `llama-server`
- La collecte de résultats dans un `DataFrame` polars et l'export en CSV
- Le cycle de vie du serveur : arrêter, relancer avec un autre modèle

## Prérequis

- `llama-server` fonctionnel (voir la partie build/install du cours)
- Deux fichiers GGUF sur le disque. Idées : deux familles différentes (Qwen3-8B vs Llama-3.1-8B), ou **le même modèle en deux quantisations** (Q4_K_M vs Q8_0)
- `pip install openai requests polars matplotlib`

## Déroulement

Un seul GPU, donc **un modèle à la fois** :

1. Lancer le serveur avec le modèle A
2. Exécuter les 50 questions → `resultats_A.csv`
3. Arrêter le serveur (`Ctrl+C`), relancer avec le modèle B
4. Ré-exécuter les 50 questions → `resultats_B.csv`
5. Charger les deux CSV et analyser

In [1]:
import glob
import time

import matplotlib.pyplot as plt
import polars as pl
import requests
from openai import OpenAI
import pandas as pd

## Étape 0 – Lancer le serveur (dans un terminal)

```bash
llama-server \
  -m /chemin/vers/modele_A.gguf \
  -c 8192 -np 1 -fa on -b 2048 -ub 512 -ngl 99 \
  --host 127.0.0.1 --port 8080
```

> **Tip** : notez quelque part le nom exact du modèle et sa quantisation (par exemple `Qwen3-8B-Q4_K_M`). Vous en aurez besoin pour nommer le fichier de résultats.

In [ ]:
BASE_URL = "http://127.0.0.1:8080"
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="pas-besoin")

# Nom court du modèle actuellement servi : à modifier à chaque relance du serveur
MODEL_NAME = "modele_A"

# Vérification que le serveur répond
print(requests.get(f"{BASE_URL}/health").json())
print([m.id for m in client.models.list().data])

## Étape 1 – Charger les questions

Le fichier `01_questions.md` contient un tableau Markdown. Chaque ligne de question commence par `|`.

Écrivez une fonction `charger_questions(chemin)` qui renvoie un `DataFrame` avec les colonnes `id`, `categorie`, `type`, `question`, `reponse_attendue`.

> **Tips**
> - Lisez le fichier ligne par ligne, ne gardez que celles qui commencent par `|`.
> - La première ligne gardée est l'en-tête, la deuxième est le séparateur `|----|` : sautez-les.
> - `ligne.strip().strip("|").split("|")` découpe une ligne en cellules ; pensez à `.strip()` chaque cellule.
> - Vérifiez que vous obtenez bien **50** lignes.

In [ ]:
def charger_questions(chemin: str) -> pl.DataFrame:
    # TODO
    ...

questions = charger_questions("01_questions.md")
print(len(questions))
questions.head()

## Étape 2 – Une fonction pour poser une question

Écrivez `poser(question: str) -> dict` qui envoie la question au modèle et renvoie un dictionnaire contenant au minimum :

- `reponse` : le texte de la réponse
- `completion_tokens` : nombre de tokens générés (`response.usage`)
- `tok_par_s` : vitesse de génération (`timings["predicted_per_second"]`)
- `duree_ms` : durée totale (`timings["prompt_ms"] + timings["predicted_ms"]`)
- `finish_reason`

> **Tips**
> - `temperature=0` pour que les résultats soient reproductibles. Pourquoi est-ce important ici ?
> - `max_tokens=500` suffit pour les questions ouvertes. Surveillez `finish_reason == "length"` pour repérer les réponses coupées.
> - Les `timings` sont un ajout de `llama-server`, le SDK les range dans `response.model_extra["timings"]`.
> - Un system prompt court aide à comparer équitablement : par exemple *« Réponds en français, de façon précise et concise. »*
> - **Qwen3** : le mode raisonnement (`<think>`) rend 50 questions très lentes. Désactivez-le avec `extra_body={"chat_template_kwargs": {"enable_thinking": False}}`.

In [ ]:
SYSTEM_PROMPT = "Réponds en français, de façon précise et concise."

def poser(question: str) -> dict:
    # TODO
    ...

# Test rapide
poser("Quelle est la capitale de l'Australie ?")

## Étape 3 – Boucle sur les 50 questions

Parcourez le `DataFrame` des questions, appelez `poser()` pour chacune, et construisez une **liste de dictionnaires** contenant les infos de la question (id, catégorie, type, réponse attendue) **et** celles de la réponse. Ajoutez une colonne `modele` avec `MODEL_NAME`.

Convertissez la liste en `DataFrame` et sauvegardez-la : `resultats_{MODEL_NAME}.csv`.

> **Tips**
> - `for q in questions.iter_rows(named=True):` donne accès à `q["question"]`, `q["id"]`, etc.
> - Affichez la progression (`print(f"{i}/50 ...")`), ça prend une à deux minutes.
> - Mesurez aussi le temps total de la boucle avec `time.perf_counter()`.
> - `df.write_csv(...)` pour sauvegarder. Polars n'a pas d'index, il n'y a donc rien à désactiver.

In [ ]:
resultats = []
debut = time.perf_counter()

# TODO : boucle

duree_totale = time.perf_counter() - debut
df = pl.DataFrame(resultats)
df.write_csv(f"resultats_{MODEL_NAME}.csv")
print(f"Terminé en {duree_totale:.0f} s → resultats_{MODEL_NAME}.csv")
df.head()

## Étape 4 – Changer de modèle

1. Dans le terminal : `Ctrl+C` pour arrêter le serveur.
2. Relancez-le avec le modèle B (même commande, autre fichier `-m`).
3. Dans ce notebook : modifiez `MODEL_NAME` dans la cellule de setup, ré-exécutez le setup et l'étape 3.

Vous devez maintenant avoir deux fichiers CSV.

## Étape 5 – Évaluation automatique des questions fermées

Chargez les deux CSV et concaténez-les (`pl.concat`). Ajoutez une colonne booléenne `correct` :

- pour les questions **fermées** : `True` si `reponse_attendue.lower()` est contenu dans `reponse.lower()`
- pour les questions **ouvertes** : `null` (pas d'évaluation automatique)

Puis calculez le **taux de bonnes réponses par modèle**, et par **modèle × catégorie**.

> **Tips**
> - `df.group_by("modele").agg(pl.col("correct").mean())`
> - `df.pivot(on="modele", index="categorie", values="correct", aggregate_function="mean")` ou `group_by(["modele", "categorie"])` pour le détail
> - `pl.when(...).then(...).otherwise(None)` permet d'écrire la condition fermée / ouverte en une expression.
> - Regardez quelques cas marqués faux : la vérification par `in` est-elle toujours juste ? (Indice : question 6, question 28…)

In [ ]:
# TODO : charger, concaténer, évaluer

## Étape 6 – Vitesse, longueur, questions ouvertes

1. Par modèle : `tok_par_s` moyen, `completion_tokens` moyen, durée totale.
2. Un graphique en barres comparant les deux modèles sur ces trois métriques.
3. Pour les questions **ouvertes**, affichez les deux réponses côte à côte pour 3 questions de votre choix et notez-les à la main (0, 1 ou 2).

> **Tip** : `pl.Config.set_fmt_str_lengths(200)` pour lire les réponses entières dans un DataFrame, ou tout simplement `print()`.

In [ ]:
# TODO

## Questions de réflexion (à répondre dans cette cellule)

1. Le modèle le plus juste est-il aussi le plus rapide ? Le plus bavard ?
2. Sur quelles catégories les deux modèles divergent-ils le plus ? Avez-vous une hypothèse ?
3. Les questions 31 et 32 (compter des lettres) sont notoirement difficiles pour les LLM. Pourquoi, d'après ce que vous savez des tokens ?
4. La vérification automatique par `in` a-t-elle produit des faux positifs ou des faux négatifs ? Donnez un exemple et proposez une amélioration.
5. Pourquoi a-t-on fixé `temperature=0` ? Que se passerait-il en relançant l'expérience avec `temperature=1` ?

*Vos réponses ici…*

## Bonus

- **Quantisation** : si vous avez comparé deux familles, refaites l'expérience avec le même modèle en Q4_K_M et Q8_0. La qualité change-t-elle ? La VRAM (`nvidia-smi`) ? La vitesse ?
- **Débit** : relancez le serveur avec `-np 4` et envoyez les questions en parallèle avec `concurrent.futures.ThreadPoolExecutor(max_workers=4)`. Comparez le temps total et le `tok_par_s` par requête avec la version `-np 1`. Que constatez-vous ?
- **Juge automatique** : pour les questions ouvertes, demandez à un troisième appel de noter la réponse de 0 à 2 en lui fournissant la question, la réponse et les éléments attendus.